# Exercises

In this section we have one exercise:
1. Implement the polynomial kernel.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [1]:
# imports
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris
import numpy as np
from sklearn.model_selection import train_test_split
import cvxopt
from sklearn.preprocessing import StandardScaler

In [2]:
# load and prepare data
iris = load_iris()
data_set = iris.data
labels = iris.target

data_set = data_set[labels!=2]
labels = labels[labels!=2]

train_data_set, test_data_set, train_labels, test_labels = train_test_split(
    data_set, labels, test_size=0.2, random_state=15)

scaler = StandardScaler()
train_data_set = scaler.fit_transform(train_data_set)
test_data_set = scaler.transform(test_data_set)

train_labels[train_labels<1] = -1
test_labels[test_labels<1] = -1

objects_count = len(train_labels)

In [3]:
# kernel building for types: linear, polynomial, radial
def build_kernel(data_set, kernel_type='linear', sigma=1.1, degree=5, coef0=1.0):
    if kernel_type == 'linear':
        return np.dot(data_set, data_set.T)
    elif kernel_type == 'poly':
        # (x·y^T + coef0)^degree
        return (np.dot(data_set, data_set.T) + coef0) ** degree
    elif kernel_type == 'rbf':
        # exp(-||x-y||^2 / (2 * sigma^2))
        X1_sq = np.sum(data_set**2, axis=1).reshape(-1, 1)
        X2_sq = np.sum(data_set**2, axis=1).reshape(1, -1)
        sq_dists = X1_sq + X2_sq - 2 * np.dot(data_set, data_set.T)
        return np.exp(-sq_dists / (2. * sigma ** 2))
    else:
        raise ValueError(f"Unknown kernel type: {kernel_type}")

In [4]:
# training
def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    cvxopt.solvers.options['show_progress'] = False
    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x'])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)))
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

In [5]:
# prediction
def classify(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id, kernel_type='linear'):
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)
    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return np.sign(y)

In [6]:
# 1. Linear
lambdas, sv, sv_id, b, targets, v_num = train(train_data_set, train_labels, kernel_type='linear')
predicted = classify(test_data_set, train_data_set, lambdas, targets, b, v_num, sv, sv_id, kernel_type='linear')
print(f'Linear: {accuracy_score(predicted, test_labels)}')

# 2. RBF
lambdas_rbf, sv_rbf, sv_id_rbf, b_rbf, targets_rbf, v_num_rbf = train(train_data_set, train_labels, kernel_type='rbf')
predicted_rbf = classify(test_data_set, train_data_set, lambdas_rbf, targets_rbf, b_rbf, v_num_rbf, sv_rbf, sv_id_rbf, kernel_type='rbf')
print(f'RBF: {accuracy_score(predicted_rbf, test_labels)}')

# 3. Polynomial
lambdas_poly, sv_poly, sv_id_poly, b_poly, targets_poly, v_num_poly = train(train_data_set, train_labels, kernel_type='poly')
predicted_poly = classify(test_data_set, train_data_set, lambdas_poly, targets_poly, b_poly, v_num_poly, sv_poly, sv_id_poly, kernel_type='poly')
print(f'Poly: {accuracy_score(predicted_poly, test_labels)}')

Linear: 0.3
RBF: 0.5
Poly: 0.75
